# 02 - Cleaning & Operasi Data
Filter, transform, dedup, join, impute, bin, aggregate. Tiap operasi dicatat row count before/after.

In [ ]:
import os, sys
import pandas as pd

BASE_DIR = os.environ.get("BDA_BASE_DIR", "..")
RAW_DIR = os.path.join(BASE_DIR, "data", "raw")
INTERIM_DIR = os.path.join(BASE_DIR, "data", "interim")
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")
FIG_DIR = os.path.join(BASE_DIR, "figures")
for d in (INTERIM_DIR, PROCESSED_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

sys.path.insert(0, os.path.join(BASE_DIR, "src"))
import cleaning as cl

op_log = []  # catatan row count before/after tiap operasi


In [ ]:
usgs_df = cl.load_usgs(RAW_DIR)
bmkg_df = cl.load_bmkg(RAW_DIR)
df = pd.concat([usgs_df, bmkg_df], ignore_index=True)

op_log.append(dict(operasi="load", before=0, after=len(df)))
print(f"USGS: {len(usgs_df)} | BMKG: {len(bmkg_df)} | total gabungan: {len(df)}")


In [ ]:
# Operasi 1: Filter/Selection - buang event luar Indonesia
before = len(df)
df = cl.filter_indonesia(df)
op_log.append(dict(operasi="filter_indonesia", before=before, after=len(df)))
print(f"Filter Indonesia: {before} -> {len(df)}")


In [ ]:
# Operasi 2: Transformation - parsing waktu, UTC<->WIB, year/month
before = len(df)
df = cl.transform_time(df)
op_log.append(dict(operasi="transform_time", before=before, after=len(df)))
print(f"Transform time: {before} -> {len(df)} (row tidak berubah, kolom bertambah)")


In [ ]:
# Operasi 3: Deduplication - event sama dari USGS & BMKG (lokasi+waktu dekat)
before = len(df)
df = cl.deduplicate(df)
op_log.append(dict(operasi="deduplicate", before=before, after=len(df)))
print(f"Dedup: {before} -> {len(df)}")


In [ ]:
# Operasi 4: Join - validasi silang USGS vs BMKG (bukan merge kolom, tapi cross-check kesamaan)
before = len(df)
df = cl.cross_validate_join(df)
op_log.append(dict(operasi="cross_validate_join", before=before, after=len(df)))
print(f"Cross-validate: {before} -> {len(df)}, {df['cross_validated'].sum()} event tervalidasi 2 sumber")


In [ ]:
# Operasi 5: Imputation - mmi/cdi/felt missing -> flag, bukan drop
before = len(df)
missing_pct = df[["mmi", "cdi", "felt"]].isna().mean() * 100
print("Persentase missing sebelum imputasi:\n", missing_pct)
df = cl.impute_missing(df)
op_log.append(dict(operasi="impute_missing", before=before, after=len(df)))


In [ ]:
# Operasi 6: Binning - depth_class, mag_band, zone_id
before = len(df)
df = cl.add_bins(df)
op_log.append(dict(operasi="add_bins", before=before, after=len(df)))
print(df[["depth_class", "mag_band"]].value_counts())


In [ ]:
# Operasi 7: Aggregation - event count & avg magnitude per zona per bulan
zone_month_agg = cl.aggregate_zone_month(df)
zone_month_agg.to_csv(os.path.join(INTERIM_DIR, "zone_month_aggregate.csv"), index=False)
op_log.append(dict(operasi="aggregate_zone_month", before=len(df), after=len(zone_month_agg)))
zone_month_agg.head()


In [ ]:
# Log ringkasan semua operasi (before -> after)
op_log_df = pd.DataFrame(op_log)
op_log_df.to_csv(os.path.join(INTERIM_DIR, "_cleaning_operation_log.csv"), index=False)
op_log_df


In [ ]:
# EDA cepat
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df["mag"].hist(bins=30, ax=axes[0])
axes[0].set_title("Distribusi Magnitude")
df["depth_km"].hist(bins=30, ax=axes[1])
axes[1].set_title("Distribusi Depth (km)")
df.groupby("year").size().plot(kind="bar", ax=axes[2])
axes[2].set_title("Jumlah Event per Tahun")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "02_eda_overview.png"), dpi=150)
plt.show()


In [ ]:
# Simpan hasil akhir - kontrak ke notebook 03/04/05
out_path = os.path.join(PROCESSED_DIR, "earthquake_features.parquet")
df.to_parquet(out_path, index=False)
print(f"Tersimpan: {out_path} ({len(df)} baris, {df.shape[1]} kolom)")
